In [3]:
import numpy as np

In [ ]:
## generated by `prompts/sbr-mlb-prompt.md`

import json
import re
from pathlib import Path
import pandas as pd

rows = []

for path in sorted(Path('sbr_scrapes/raw_data').glob('*.json')):
    match = re.search(r'(\d{4}-\d{2}-\d{2})', path.stem)
    date = match.group(1) if match else None

    with path.open() as f:
        data = json.load(f)

    for game in data.get('games', []):
        row = {k: v for k, v in game.items() if k != 'odds'}
        row['date'] = date
        for book in game.get('odds', []):
            name = book['name']
            ## added by me
            if not name.startswith("Fanatics"):
                row[f'{name}_team1_odds'] = book.get('team1_odds')
                row[f'{name}_team2_odds'] = book.get('team2_odds')
        rows.append(row)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])

# move date to the front
cols = ['date'] + [c for c in df.columns if c != 'date']
df = df[cols]

df.iloc[0]

date                        2026-03-26 00:00:00
game_time                2026-03-26 1:15 PM EDT
team1_name                                  PIT
team2_name                                  NYM
team1_score                                 7.0
team2_score                                11.0
team1_pitcher                         P. Skenes
team2_pitcher                        F. Peralta
team1_pitcher_hand                            R
team2_pitcher_hand                            R
team1_percent_wagers                         47
team2_percent_wagers                         53
team1_opener_odds                           115
team2_opener_odds                          -135
BetMGM_team1_odds                         105.0
BetMGM_team2_odds                        -115.0
FanDuel_team1_odds                        100.0
FanDuel_team2_odds                       -118.0
Caesars_team1_odds                        105.0
Caesars_team2_odds                       -125.0
bet365_team1_odds                       

In [5]:
## FIXME: ugly copy-paste from scrape_yahoo_odds/scrape_utils.py

def convert_line(line):
    """
    convert American style money line to the implied probability

    -400 implies you will win 4 out of 5 bets
    >>> convert_line(-400)
    0.8
    >>> convert_line(+300)
    0.25
    """
    if line < 0:
        return abs(line)/(abs(line)+100)
    else:
        return 100/(100+line)
    
def payout(line):
    """
    calculates amount of profit from taking an American style money line (risking $100)
    
    the 2 times you win a -200 bet covers the 1 time you lose it, so payout should be $50.0
    >>> payout(-200)
    50.0

    a +300 bet should payout $300
    >>> payout(300)
    300.0
    """
    if pd.isnull(line):
        return None
    return (100/convert_line(line)) - 100

there's a problem with using median/mean for these due to -100 and +100 being the same thing (which averages to zero)

In [6]:
df[['FanDuel_team1_odds', 'Caesars_team1_odds', 
                              'bet365_team1_odds', 'DraftKings_team1_odds']].map(convert_line).median(axis=1)

0      0.497525
1      0.380954
2      0.363636
3      0.452498
4      0.390649
         ...   
918    0.320020
919    0.535960
920    0.454545
921    0.513379
922    0.575370
Length: 923, dtype: float64

mapping these to the implied odds then taking the median (can't take median of american style lines because -100/+100 are the same thing)

In [7]:
df['median_team1_implied'] = df[['FanDuel_team1_odds', 'Caesars_team1_odds', 
                              'bet365_team1_odds', 'DraftKings_team1_odds']].map(convert_line).median(axis=1)

df['median_team2_implied'] = df[['FanDuel_team2_odds', 'Caesars_team2_odds', 
                              'bet365_team2_odds', 'DraftKings_team2_odds']].map(convert_line).median(axis=1)

In [8]:
df['overround'] = df['median_team1_implied'] + df['median_team2_implied']

ok, so I'm calculating the implied odds correctly, but there is some fishy data

In [9]:
wtf = df[df.overround < 1].copy()

In [22]:
len(wtf)

90

note how draftkings is at -102/+118 for this game. I don't know who's responsible for the bad data but this would never happen in reality

In [10]:
convert_line(-102) + convert_line(118)

0.9636660913797802

In [11]:
wtf.iloc[2]

date                     2026-04-01 00:00:00
game_time                       12:35 PM EDT
team1_name                               TEX
team2_name                               BAL
team1_score                              NaN
team2_score                              NaN
team1_pitcher                     N. Eovaldi
team2_pitcher                      T. Rogers
team1_pitcher_hand                         R
team2_pitcher_hand                         L
team1_percent_wagers                      44
team2_percent_wagers                      56
team1_opener_odds                       -105
team2_opener_odds                       -115
BetMGM_team1_odds                     -105.0
BetMGM_team2_odds                      115.0
FanDuel_team1_odds                    -102.0
FanDuel_team2_odds                     116.0
Caesars_team1_odds                     100.0
Caesars_team2_odds                     120.0
bet365_team1_odds                      100.0
bet365_team2_odds                      120.0
DraftKings

HERE: there's some bug with the ETL code...the implied odds should always be over 1.0

there are ones with median_team2_odds equal to eg -1.0, -2.0

In [12]:
df[df.overround < 1.0]
#.groupby('date').count()

,date,game_time,team1_name,team2_name,team1_score,team2_score,team1_pitcher,team2_pitcher,team1_pitcher_hand,team2_pitcher_hand,...,FanDuel_team2_odds,Caesars_team1_odds,Caesars_team2_odds,bet365_team1_odds,bet365_team2_odds,DraftKings_team1_odds,DraftKings_team2_odds,median_team1_implied,median_team2_implied,overround
31,2026-03-28,8:40 PM EDT,DET,SD,0.0,3.0,J. Flaherty,R. Vasquez,R,R,...,108.0,-105.0,115.0,-110.0,110.0,-108.0,112.0,0.519231,0.473944,0.993175
75,2026-04-01,12:15 PM EDT,ATH,ATL,NaN,NaN,L. Severino,C. Sale,R,L,...,230.0,192.0,235.0,195.0,240.0,189.0,232.0,0.343647,0.299856,0.643503
76,2026-04-01,12:35 PM EDT,TEX,BAL,NaN,NaN,N. Eovaldi,T. Rogers,R,L,...,116.0,100.0,120.0,100.0,120.0,-102.0,118.0,0.502475,0.456631,0.959106
113,2026-04-04,4:10 PM EDT,SD,BOS,3.0,2.0,R. Vasquez (R),C. Early (L),R,L,...,144.0,122.0,145.0,125.0,150.0,123.0,149.0,0.449440,0.404885,0.854325
114,2026-04-04,7:05 PM EDT,MIA,NYY,7.0,9.0,M. Meyer (R),R. Weathers (L),R,L,...,196.0,162.0,195.0,165.0,200.0,163.0,199.0,0.379508,0.336143,0.715651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,2026-05-29,7:10 PM EDT,BOS,CLE,3.0,4.0,T. Samaniego,S. Cecconi,L,R,...,126.0,105.0,125.0,105.0,125.0,104.0,126.0,0.487805,0.443461,0.931266
863,2026-05-31,1:40 PM EDT,BOS,CLE,9.0,4.0,R. Suarez,T. Bibee,L,R,...,116.0,-105.0,115.0,100.0,120.0,-105.0,114.0,0.508573,0.464040,0.972612
873,2026-05-31,7:20 PM EDT,CHC,STL,1.0,5.0,J. Wicks,M. Liberatore,L,L,...,118.0,100.0,120.0,100.0,120.0,-101.0,120.0,0.500000,0.454545,0.954545
887,2026-06-02,6:45 PM EDT,MIA,WAS,7.0,3.0,L. Bachar,R. Lovelady,R,L,...,116.0,-105.0,115.0,-105.0,115.0,-106.0,114.0,0.512195,0.465116,0.977311


In [13]:
gb = df[df.overround < 1.0].groupby("date").count().game_time
gb

date
2026-03-28     1
2026-04-01     2
2026-04-04     5
2026-04-05     2
2026-04-06     1
2026-04-11     7
2026-04-14     1
2026-04-17     2
2026-04-18     1
2026-04-19     2
2026-04-20     1
2026-04-21     1
2026-04-23     1
2026-04-24     1
2026-04-25     2
2026-04-27     1
2026-04-28     8
2026-04-30     5
2026-05-03     3
2026-05-07     1
2026-05-09     5
2026-05-10    13
2026-05-12     2
2026-05-16     2
2026-05-17     2
2026-05-18     1
2026-05-19     5
2026-05-21     2
2026-05-22     1
2026-05-23     1
2026-05-25     1
2026-05-26     1
2026-05-29     2
2026-05-31     2
2026-06-02     1
2026-06-03     1
Name: game_time, dtype: int64

are there really 36 days with parse errors?

In [14]:
gb.shape

(36,)

90 games with problems (at least, there could be other problems as well)

900 ish games total, so that's about a 10% failure rate

In [15]:
len(df[df.overround < 1.0])

90

In [16]:
ok_df = df[df.overround > 1].copy()

there are still a few fishy overrounds. they should all be close to 1.04

In [17]:
ok_df.overround.describe()

count    778.000000
mean       1.041953
std        0.005439
min        1.001139
25%        1.041002
50%        1.042547
75%        1.044135
max        1.051637
Name: overround, dtype: float64

the opening lines on all these are fine

In [27]:
ok_df.loc[ok_df.overround < 1.03,["date", "team1_name",
                                  "team2_name", "overround", 
                                  "team1_opener_odds",
                                  "team2_opener_odds",
                                  "median_team1_implied", 
                                  "median_team2_implied",
                                  ]]

,date,team1_name,team2_name,overround,team1_opener_odds,team2_opener_odds,median_team1_implied,median_team2_implied
112,2026-04-04,BAL,PIT,1.011052,-115,-105,0.502475,0.508576
120,2026-04-04,NYM,SF,1.022689,-125,105,0.534884,0.487805
142,2026-04-06,STL,WAS,1.022689,-110,-110,0.534884,0.487805
264,2026-04-15,SF,CIN,1.002289,-120,100,0.523810,0.478480
269,2026-04-15,TB,CHW,1.022689,-118,-102,0.534884,0.487805
271,2026-04-15,SEA,SD,1.002289,-110,-110,0.523810,0.478480
303,2026-04-18,SF,WAS,1.008053,-120,100,0.526056,0.481998
313,2026-04-18,SD,LAA,1.019398,-115,-105,0.531593,0.487805
342,2026-04-21,CIN,TB,1.022689,-115,-105,0.534884,0.487805
371,2026-04-23,PHI,CHC,1.022689,-130,110,0.534884,0.487805


Turns out some of the "ok" ones have wrong data as well, eg. 4-29. but the game in question isn't in the df! Why?

In [19]:
ok_df[(ok_df.team1_name == "LAA") & (ok_df.team2_name == "CHW")]

,date,game_time,team1_name,team2_name,team1_score,team2_score,team1_pitcher,team2_pitcher,team1_pitcher_hand,team2_pitcher_hand,...,FanDuel_team2_odds,Caesars_team1_odds,Caesars_team2_odds,bet365_team1_odds,bet365_team2_odds,DraftKings_team1_odds,DraftKings_team2_odds,median_team1_implied,median_team2_implied,overround
415,2026-04-27,7:40 PM EDT,LAA,CHW,7.0,8.0,J. Kochanowicz,A. Kay,R,L,...,108.0,100.0,-120.0,100.0,-120.0,-102.0,118.0,0.502475,0.513112,1.015587
427,2026-04-28,7:40 PM EDT,LAA,CHW,2.0,5.0,J. Soriano,D. Martin,R,R,...,110.0,-130.0,110.0,-130.0,110.0,-137.0,114.0,0.565217,0.476190,1.041408


is it in the original df?

In [20]:
df[(df.team1_name == "LAA") & (df.team2_name == "CHW")]

,date,game_time,team1_name,team2_name,team1_score,team2_score,team1_pitcher,team2_pitcher,team1_pitcher_hand,team2_pitcher_hand,...,FanDuel_team2_odds,Caesars_team1_odds,Caesars_team2_odds,bet365_team1_odds,bet365_team2_odds,DraftKings_team1_odds,DraftKings_team2_odds,median_team1_implied,median_team2_implied,overround
415,2026-04-27,7:40 PM EDT,LAA,CHW,7.0,8.0,J. Kochanowicz,A. Kay,R,L,...,108.0,100.0,-120.0,100.0,-120.0,-102.0,118.0,0.502475,0.513112,1.015587
427,2026-04-28,7:40 PM EDT,LAA,CHW,2.0,5.0,J. Soriano,D. Martin,R,R,...,110.0,-130.0,110.0,-130.0,110.0,-137.0,114.0,0.565217,0.476190,1.041408
435,2026-04-29,1:10 PM EDT,LAA,CHW,2.0,3.0,Y. Kikuchi,E. Fedde,L,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


ok, so there are a bunch of them with NaN. I checked the page itself and those odds are there.

In [21]:
df.iloc[435]

date                     2026-04-29 00:00:00
game_time                        1:10 PM EDT
team1_name                               LAA
team2_name                               CHW
team1_score                              2.0
team2_score                              3.0
team1_pitcher                     Y. Kikuchi
team2_pitcher                       E. Fedde
team1_pitcher_hand                         L
team2_pitcher_hand                         R
team1_percent_wagers                      43
team2_percent_wagers                      57
team1_opener_odds                       -130
team2_opener_odds                        110
BetMGM_team1_odds                     -125.0
BetMGM_team2_odds                      105.0
FanDuel_team1_odds                       NaN
FanDuel_team2_odds                       NaN
Caesars_team1_odds                       NaN
Caesars_team2_odds                       NaN
bet365_team1_odds                        NaN
bet365_team2_odds                        NaN
DraftKings

there are also some with "No content available". the ones I looked at have "-" for some of the lines. I made the odds nullable in the schema, so that might fix it.

In [26]:
df[df.FanDuel_team1_odds.isna()]

,date,game_time,team1_name,team2_name,team1_score,team2_score,team1_pitcher,team2_pitcher,team1_pitcher_hand,team2_pitcher_hand,...,FanDuel_team2_odds,Caesars_team1_odds,Caesars_team2_odds,bet365_team1_odds,bet365_team2_odds,DraftKings_team1_odds,DraftKings_team2_odds,median_team1_implied,median_team2_implied,overround
166,2026-04-08,No content available,No content available,No content available,0.0,0.0,No content available,No content available,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223,2026-04-12,1:40 PM EDT,MIA,DET,2.0,8.0,S. Alcantara,T. Skubal,R,L,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
224,2026-04-12,1:40 PM EDT,LAA,CIN,9.0,6.0,J. Soriano,A. Abbott,R,L,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,2026-04-12,2:10 PM EDT,CHW,KC,6.0,5.0,G. Taylor,N. Cameron,R,L,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,2026-04-12,2:10 PM EDT,WAS,MIL,8.0,6.0,Z. Littell,B. Woodruff,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227,2026-04-12,2:15 PM EDT,BOS,STL,9.0,3.0,B. Bello,A. Pallante,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,2026-04-12,2:20 PM EDT,PIT,CHC,6.0,7.0,B. Chandler,J. Taillon,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
229,2026-04-12,4:10 PM EDT,COL,SD,2.0,7.0,J. Herget,N. Pivetta,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
230,2026-04-12,4:10 PM EDT,HOU,SEA,1.0,6.0,C. Bolton,L. Gilbert,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
231,2026-04-12,4:10 PM EDT,TEX,LAD,5.0,2.0,J. deGrom,R. Sasaki,R,R,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
